In [1]:
from mini_competition.data_functions import load_data, add_zero_sales, split_train_set, write_result, PROCCESED_PATH, EXTRACTED_PATH, PROCCESED_PATH
import pandas as pd

data = load_data(PROCCESED_PATH / "train.parquet" )
data = data[data["date"] > "2016-07-01"]

In [2]:
# add zeros
data = add_zero_sales(data)

In [3]:
import numpy as np

# add lag features
def add_lag_features(df : pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)
    df["log_sales"] = np.log1p(df["unit_sales"].astype("float32"))

    g = df.groupby(["store_nbr", "item_nbr"])["log_sales"]

    for lag in [16, 21, 28, 35]:
        df[f"lag_{lag}"] = g.shift(lag)

    df["rolling_mean_7"]  = g.transform(lambda s: s.shift(16).rolling(7).mean())
    df["rolling_mean_28"] = g.transform(lambda s: s.shift(16).rolling(28).mean())

    return df.copy().drop(columns=["log_sales"])
    
# add lag features
data_with_lags = add_lag_features(data)

# drop none rows
cols = ["lag_16", "lag_21","lag_28", "lag_35","rolling_mean_7", "rolling_mean_28"]
data_with_lags = data_with_lags.dropna(subset=cols, how="any")

print(data_with_lags.info())


<class 'pandas.DataFrame'>
Index: 79767504 entries, 43 to 89139095
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   date             datetime64[ms]
 1   store_nbr        int8          
 2   item_nbr         int32         
 3   unit_sales       float32       
 4   onpromotion      object        
 5   lag_16           float32       
 6   lag_21           float32       
 7   lag_28           float32       
 8   lag_35           float32       
 9   rolling_mean_7   float64       
 10  rolling_mean_28  float64       
dtypes: datetime64[ms](1), float32(5), float64(2), int32(1), int8(1), object(1)
memory usage: 4.8+ GB
None


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import TargetEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np

def clean_features(d):
    d = d.copy()
    d["onpromotion"] = d["onpromotion"].fillna(False)
    return d

def add_date_features(d):
    d = d.copy()
    # adds weekday and month feature
    d["date"] = pd.to_datetime(d["date"])
    d["weekday_num"] = d["date"].dt.day_of_week
    d["monthday_num"] = d["date"].dt.day
    d["month_num"] = d["date"].dt.month
    return d.drop(columns="date")

df = data_with_lags

(x_train, y_train, x_valid, y_valid) = split_train_set(df, "2017-07-31")

# Target-encode item_nbr and pass the other columns through unchanged
preprocess = ColumnTransformer(
    [("item_te", TargetEncoder(), ["item_nbr"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")   # keeps store_nbr as 'category'

model = TransformedTargetRegressor(
    regressor=make_pipeline(
        FunctionTransformer(clean_features),
        FunctionTransformer(add_date_features),
        preprocess,
        HistGradientBoostingRegressor(
            categorical_features=["store_nbr"],
            max_iter=1000, learning_rate=0.05, max_leaf_nodes=127,
            early_stopping=True,
        ),
    ),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,   # clipping can't be undone, so skip the round-trip check
)


model.fit(x_train, y_train)
pred = model.predict(x_valid)

In [7]:
np.mean(pred)

np.float64(2.9683396939319397)

In [10]:
# scoring

from sklearn.metrics import root_mean_squared_log_error

error = root_mean_squared_log_error(pred, y_valid)
print(f"Error for random sample set is: {error}")

Error for random sample set is: 0.5672854054636907


In [ ]:
# load test set and add lag
from pathlib import Path

test_data = load_data(EXTRACTED_PATH / "test.csv", dtype={"id": int})

test_data["unit_sales"] = None

# combine with previous history
combined = pd.concat([df[df["date"] >= "2017-06-01"], test_data], ignore_index=True)

# add zero sales and then lag features
combined_with_zeros = add_zero_sales(combined)
combined_with_lags = add_lag_features(combined_with_zeros)

combined_with_lags[combined_with_lags["id"] > 0].info()

<class 'pandas.DataFrame'>
Index: 3370464 entries, 76 to 20348927
Data columns (total 12 columns):
 #   Column           Dtype  
---  ------           -----  
 0   date             object 
 1   store_nbr        int64  
 2   item_nbr         int64  
 3   unit_sales       object 
 4   onpromotion      object 
 5   lag_16           float32
 6   lag_21           float32
 7   lag_28           float32
 8   lag_35           float32
 9   rolling_mean_7   float64
 10  rolling_mean_28  float64
 11  id               float64
dtypes: float32(4), float64(3), int64(2), object(3)
memory usage: 282.9+ MB


In [12]:
# create csv
import os
from pathlib import Path

EXTRACTED_FOLDER = Path("./../data/raw") / "extracted"
RESULT_FOLDER = Path("./../data/results")

test_set = combined_with_lags[combined_with_lags["id"] > 0]

# make predictions
features = test_set.columns.drop(["unit_sales", "id"])
test_pred = model.predict(test_set[features])

# clip to 0 as minimum
test_pred_positive = np.clip(test_pred, 0, None)

submission_data = pd.DataFrame({
    "id": test_set["id"].to_numpy().astype("int64"),
    "unit_sales": np.asarray(test_pred_positive, dtype=float).ravel(),
})

write_result(submission_data, "lag_features_result_v2.csv")

